# `__slots__` и исключения в ООП – теория

## 1. `__slots__` – оптимизация памяти и ограничение атрибутов

По умолчанию каждый объект Python имеет словарь `__dict__`, в котором хранятся его атрибуты. Это удобно, но требует дополнительной памяти. `__slots__` позволяет:

- **Фиксировать набор атрибутов** – объекты с `__slots__` не имеют `__dict__`
- **Экономить память** – атрибуты хранятся не в словаре, а в дескрипторах (как в C-структуре)
- **Ускорить доступ** – доступ к атрибутам через `__slots__` быстрее

**Синтаксис:**

```python
class Point:
    __slots__ = ('x', 'y')
    def __init__(self, x, y):
        self.x = x
        self.y = y

**Важные ограничения:**

- Нельзя динамически добавлять новые атрибуты (если только `__slots__` не включает `'__dict__'`)
- Множественное наследование с `__slots__` требует осторожности (каждый класс определяет свои `__slots__`)
- Объекты с `__slots__` не поддерживают слабые ссылки, если не добавить `'__weakref__'` в `__slots__`
- При наследовании дочерний класс получает `__dict__`, если в его `__slots__` не перечислены атрибуты родителя

## 2. Исключения в ООП

Исключения – механизм для обработки ошибок и нештатных ситуаций.

### Иерархия

- `BaseException` – корень всех исключений
  - `SystemExit`, `KeyboardInterrupt` – системные
  - `Exception` – базовый для всех обычных ошибок
    - `ValueError`, `TypeError`, `IndexError`, `ArithmeticError` и т.д.

**Пользовательские исключения** создаются наследованием от `Exception` (или его подклассов):

In [ ]:
class ValidationError(Exception):
    pass

In [ ]:
try:
    # код, который может выбросить исключение
except ValueError as e:
    # обработка конкретного исключения
except (ZeroDivisionError, TypeError):
    # обработка нескольких типов
except:
    # плохая практика – перехватывает всё, включая SystemExit
else:
    # выполняется, если исключения не было
finally:
    # выполняется всегда (закрытие ресурсов)

## 🟢 Базовые задачи

### Задача 1. Класс с `__slots__`
Создайте класс `Student` с `__slots__ = ('name', 'age', 'grade')`. Реализуйте конструктор и метод `__str__`. Создайте несколько студентов. Попробуйте динамически добавить атрибут `extra`. Выведите размер объекта через `sys.getsizeof()` и сравните с классом без `__slots__`.

In [ ]:
# Шаблон для задачи 1
import sys

class Student:
    __slots__ = ...  # укажите атрибуты
    def __init__(self, name, age, grade):
        # инициализация
        pass

    def __str__(self):
        # вернуть строковое представление
        pass

# Создание объектов
s1 = Student("Alice", 20, 95)
# Попытка добавить атрибут
# s1.extra = "test"  # раскомментировать для проверки

# Вывод размера
print(sys.getsizeof(s1))

# Класс без __slots__ для сравнения
class RegularStudent:
    def __init__(self, name, age, grade):
        self.name = name
        self.age = age
        self.grade = grade

r1 = RegularStudent("Bob", 21, 90)
print(sys.getsizeof(r1))  # плюс размер __dict__
print(sys.getsizeof(r1.__dict__))

### Задача 2. Пользовательское исключение
Создайте класс `BankAccount`. Реализуйте методы `deposit(amount)` и `withdraw(amount)`. Определите пользовательские исключения: `NegativeAmountError` (наследник `ValueError`) и `InsufficientFundsError` (наследник `ValueError`). При попытке внести отрицательную сумму или снять больше баланса выбрасывайте соответствующие исключения. В основном коде обработайте их.

In [ ]:
# Шаблон для задачи 2
class NegativeAmountError(ValueError):
    pass

class InsufficientFundsError(ValueError):
    pass

class BankAccount:
    def __init__(self, balance=0):
        self._balance = balance

    @property
    def balance(self):
        return self._balance

    def deposit(self, amount):
        if amount < 0:
            raise ...
        self._balance += amount

    def withdraw(self, amount):
        if amount < 0:
            raise ...
        if amount > self._balance:
            raise ...
        self._balance -= amount

# Проверка
acc = BankAccount(100)
try:
    acc.withdraw(200)
except InsufficientFundsError as e:
    print(e)

### Задача 3. Наследование исключений
Создайте иерархию исключений для библиотеки: `LibraryError` → `BookNotFoundError`, `BookAlreadyBorrowedError`. Класс `Library` имеет методы `add_book(title)`, `borrow_book(title)`, `return_book(title)`. При отсутствии книги – `BookNotFoundError`, при попытке взять уже выданную – `BookAlreadyBorrowedError`.

In [ ]:
# Шаблон для задачи 3
class LibraryError(Exception):
    pass

class BookNotFoundError(LibraryError):
    pass

class BookAlreadyBorrowedError(LibraryError):
    pass

class Library:
    def __init__(self):
        self._books = {}  # title -> borrowed (bool)

    def add_book(self, title):
        self._books[title] = False

    def borrow_book(self, title):
        if title not in self._books:
            raise ...
        if self._books[title]:
            raise ...
        self._books[title] = True

    def return_book(self, title):
        if title not in self._books:
            raise ...
        self._books[title] = False

# Демонстрация
lib = Library()
lib.add_book("1984")
try:
    lib.borrow_book("1984")
    lib.borrow_book("1984")
except BookAlreadyBorrowedError as e:
    print(e)


### Задача 4. Класс `SafeVector` (вектор целых чисел)
Реализуйте класс `SafeVector`, который хранит список целых чисел.  
`__slots__ = ('_data', '_size')`, где `_data` – список, `_size` – текущая длина (можно вычислять динамически, но для простоты храните отдельно).  
Методы:
- `__init__(self, initial_data=None)` – если передан список, проверяет, что все элементы – целые числа (иначе `TypeError`).
- `__getitem__(self, index)` – возвращает элемент по индексу; при выходе за границы генерирует `IndexError`.
- `append(self, value)` – добавляет значение в конец, проверяя, что `value` – целое число.
- `__setitem__(self, index, value)` – изменяет существующий элемент; проверяет тип и границы индекса.

Любая попытка присвоить атрибут, не входящий в `__slots__` (например, `safe.max_value = 100`), должна завершаться `AttributeError` (автоматически).


### Задача 5. Класс `Transaction` (валютная транзакция)
Создайте класс `Transaction` с `__slots__ = ('amount', 'currency', 'timestamp')`.  
- `amount` – положительное число (проверка на `value > 0`, иначе `ValueError`).
- `currency` – строка из трёх заглавных букв, должна входить в предопределённый набор допустимых валют: `{"USD", "EUR", "RUB", "GBP", "CNY"}`.
- `timestamp` – объект `datetime.datetime` (по умолчанию `datetime.now()`).

Определите пользовательское исключение `InvalidCurrencyError`.  
При попытке установить `currency` не из набора или строку не из трёх букв – генерируйте `InvalidCurrencyError`.  
Переопределите метод `__setattr__`, чтобы перехватывать попытки присвоения недопустимых атрибутов (тех, которых нет в `__slots__`). Вместо стандартного `AttributeError` выбрасывайте более информативное сообщение: `"Attribute 'xxx' not allowed. Use one of: amount, currency, timestamp"`.  
(Указание: для совместимости с `__slots__` внутри `__setattr__` нужно вызывать `object.__setattr__(self, name, value)` для разрешённых имён.)


### Задача 5a. Класс `Book` (год издания)
Создайте класс `Book` с `__slots__ = ('title', 'author', 'year')`.  
Конструктор принимает три параметра.  
При установке поля `year` проверяйте:
- значение должно быть целым числом (`int`);
- год должен лежать в диапазоне от 1450 (примерная дата книгопечатания) до текущего года (можно использовать `datetime.date.today().year`).

Если год не в диапазоне – генерируйте `ValueError` с сообщением вида: `"Год 2025 выходит за допустимый диапазон 1450–2024"` (пример).  
Добавьте метод `from_csv_line(cls, line: str)`, который принимает строку вида `"Название;Автор;2001"`, разбирает её и возвращает объект `Book`. При разборе обрабатывайте возможное отсутствие третьего элемента (выбрасывайте `IndexError` с понятным текстом).



## 🏠 Домашнее задание

### Задача 6. `__slots__` во вложенном классе
Создайте класс `Point3D` с `__slots__ = ('x', 'y', 'z')` и методом `distance_to(other)`. Создайте класс `PointCloud`, который хранит список точек. У `PointCloud` не должно быть `__slots__` (или может быть – решите сами). Реализуйте `add_point(x, y, z)`. Измерьте память для 100 000 точек (через `tracemalloc`). Напишите вторую версию `Point3D` без `__slots__` и сравните.

### Задача 7. Исключения и контекстный менеджер
Создайте контекстный менеджер `IgnoringErrors`, который принимает в конструкторе кортеж типов исключений. В `__exit__` подавляет эти исключения (возвращает `True`). Продемонстрируйте работу: внутри блока `with` несколько ошибок, но они игнорируются, программа продолжает выполнение.

### Задача 8. Комбинация `__slots__` и исключений
Создайте класс `ImmutablePoint` с `__slots__ = ('_x', '_y')`. Сделайте атрибуты `x` и `y` доступными только для чтения через `@property`. Переопределите `__setattr__` так, чтобы при попытке изменить `_x` или `_y` после инициализации выбрасывалось `AttributeError` с сообщением "Immutable attribute". Покажите работу.